#### **1️⃣ What exactly are Seeds in dbt?**

**dbt Seeds are CSV files that dbt loads into your data warehouse as tables.**

👉 In simple words:

> **Seeds = static data files (CSV) → version-controlled → loaded as tables → referenced in models**

They are **not SQL models**, but once loaded, they behave **exactly like tables** in your warehouse.

------------

#### **2️⃣ Why does dbt even need Seeds?**

Let’s first understand the **problem seeds solve**.

**❌ Problem without seeds**

Imagine you need:

- Country codes

- Status mappings

- Hard-coded business rules

- Small lookup tables


Without seeds, you would:

- Manually create tables in Snowflake / Postgres

- Or write long CASE WHEN logic inside models

- Or depend on external systems for static data


This causes:

- ❌ Logic spread everywhere

- ❌ Hard to track changes

- ❌ No version control

- ❌ Environment mismatch (DEV vs PROD)

--------------

**✅ What seeds fix**

Seeds allow you to:

- Store small static datasets **inside dbt**

- Version-control them with Git

- Load them consistently across environments

- Use them like normal tables

**dbt philosophy:**

> “If data is part of transformation logic, it belongs in dbt.”

---------

#### **3️⃣ Where do Seeds live in a dbt project?**

Inside your dbt project:

In [ ]:
airbnb/
│
├── seeds/
│   ├── country_codes.csv
│   ├── property_types.csv
│   └── review_status.csv

**📌 Important rules**

- Seeds **must be CSV files**

- They must live inside the seeds/ directory

- Column names come from CSV headers

-----------

#### **4️⃣ How dbt Seeds actually work (internally)**

When you run:

In [ ]:
dbt seed

dbt does the following:

- Reads all CSV files from `seeds/`

- Infers column names and data types

- Creates tables in your warehouse

- Loads data from CSV into those tables

So effectively:

In [ ]:
CSV file  →  dbt seed  →  Warehouse table

------

#### **5️⃣ Simple example (step-by-step)**

**🎯 Use case: Country codes mapping**

**Step 1: Create a seed file**

`📄 seeds/country_codes.csv`

In [ ]:
country_code,country_name,region
IN,India,APAC
US,United States,NA
GB,United Kingdom,EMEA
AU,Australia,APAC

**Step 2: Run seed command**

In [ ]:
dbt seed

Output (simplified):

In [ ]:
Loaded seed file country_codes

Created table dev.country_codes

**Step 3: Result in warehouse**

A table is created:

In [ ]:
SELECT * FROM dev.country_codes;

| country_code | country_name   | region |
| ------------ | -------------- | ------ |
| IN           | India          | APAC   |
| US           | United States  | NA     |
| GB           | United Kingdom | EMEA   |
| AU           | Australia      | APAC   |

🎉 Now this is a **real table.**

----------

#### **6️⃣ Using Seeds inside dbt models**

Seeds are referenced using `ref()`, just like models.

**Example model**

`📄 models/dim/dim_countries.sql`

In [ ]:
SELECT
    country_code,
    country_name,
    region
FROM {{ ref('country_codes') }}

📌 Important:

- `ref('country_codes')` points to the **seed table**

- dbt automatically tracks lineage

-----------

#### **7️⃣ Seeds vs Models (VERY IMPORTANT)**

| Aspect          | Seeds            | Models                   |
| --------------- | ---------------- | ------------------------ |
| Source          | CSV files        | SQL queries              |
| Data type       | Static           | Dynamic                  |
| Size            | Small            | Medium / Large           |
| Logic           | No logic         | Transformation logic     |
| Use case        | Lookup / mapping | Business transformations |
| Version control | ✅ Yes            | ✅ Yes                    |


💡 Rule of thumb:

- **Static lookup data → Seed**

- **Derived / transformed data → Model**

--------

#### **8️⃣ Real-world dbt seed use cases (very common)**

**✅ 1. Status mappings**

In [ ]:
status_id,status_name,is_active
1,active,true
2,inactive,false
3,suspended,false

Used to standardize statuses across models.

**✅ 2. Business rules**

In [ ]:
rating,min_score,max_score
poor,0,2
average,3,4
excellent,5,5

**✅ 3. Country / currency / timezone mappings**

In [ ]:
currency_code,currency_name
INR,Indian Rupee
USD,US Dollar
EUR,Euro

**✅ 4. Feature flags / config data**

In [ ]:
feature_name,is_enabled
new_checkout,true
beta_dashboard,false

-------

#### **9️⃣ Why NOT use seeds for large data?**

Seeds are **not meant for big data.**

**❌ Don’t use seeds when:**

- Data is large (100k+ rows)

- Data changes frequently

- Data comes from source systems

- Data is transactional

Why?

- CSV loading is slow

- No incremental support

- Manual updates required

---------------

**🔟 Seed configuration (advanced but important)**

You can configure seeds in dbt_project.yml:

In [ ]:
seeds:
  airbnb:
    +schema: seed_data
    +quote_columns: false

This means:

- All seeds go into `seed_data` schema

- Column names are not quoted

---------

#### **1️⃣1️⃣ How dbt detects seed changes**

If you modify the CSV:

- Add row

- Change value

- Delete row

Then run:

In [ ]:
dbt seed

dbt will:

- **TRUNCATE + RELOAD** the table (by default)

📌 Seeds are **not incremental.**
